# Variance Reduction Techniques Monte Carlo 2/2

### Black & Scholes

In [15]:
import numpy as np
from scipy.stats import norm

def black_scholes(S, K, T, r, sigma, option_type='call'):
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    if option_type == 'call':
        return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
    else:
        return K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)

### Standard Monte Carlo

In [11]:
def european_mc(S, K, T, r, sigma, n_sim, option_type='call', seed=32):
    np.random.seed(seed)
    sign = 1 if option_type == 'call' else -1

    Z = np.random.standard_normal(n_sim) # Generate random value from N(0, 1)
    S_T = S * np.exp((r - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * Z)
    payoffs = np.maximum(sign * (S_T - K), 0)

    price = np.exp(-r * T) * payoffs.mean()
    se = np.exp(-r * T) * payoffs.std(ddof=1) / np.sqrt(n_sim)
    return price, se

### Stratified Sampling

In [11]:
def european_mc_ss(S, K, T, r, sigma, n_sim, n_strata, option_type='call', seed=32):
    np.random.seed(seed)
    sign = 1 if option_type == 'call' else -1
    n_per = n_sim // n_strata

    strata = np.arange(n_strata).reshape(-1, 1) # shape (K, 1)
    V = np.random.uniform(size=(n_strata, n_per))  # shape (K, n_per)
    U = (strata + V) / n_strata # formula (13.2)
    Z = norm.ppf(U)

    S_T = S * np.exp((r - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * Z) # shape (K, n_per)
    payoffs = np.maximum(sign * (S_T - K), 0) # shape (K, n_per)

    stratum_means = payoffs.mean(axis=1) # shape (K, 1)
    stratum_vars = payoffs.var(axis=1, ddof=1) # shape (K, 1)

    price = np.exp(-r * T) * stratum_means.mean()
    se = np.exp(-r * T) * np.sqrt(stratum_vars.mean() / n_sim)
    return price, se

In [12]:
# Parameters
S, K, T, r, sigma, n_sim = 100.0, 100.0, 1.0, 0.05, 0.20, 10000
n_strata = 100

In [13]:
# Pricing
bs_price = black_scholes(S, K, T, r, sigma, 'call')
price_std, se_std = european_mc(S, K, T, r, sigma, n_sim, 'call')
price_strat, se_strat = european_mc_ss(S, K, T, r, sigma, n_sim, n_strata, 'call')

print(f"Black-Scholes                  : {bs_price:.4f}")
print(f"Standard MC   (n={n_sim:})        : {price_std:.4f} ± {1.96*se_std:.4f}")
print(f"Stratified MC (n={n_sim:}, K={n_strata}) : {price_strat:.4f} ± {1.96*se_strat:.4f}")
print(f"Variance reduction factor      : {(se_std/se_strat)**2:,.0f}x")

Black-Scholes                  : 10.4506
Standard MC   (n=10000)        : 10.6350 ± 0.2906
Stratified MC (n=10000, K=100) : 10.4535 ± 0.0243
Variance reduction factor      : 143x


### Importance Sampling

In [14]:
def european_mc_is(S, K, T, r, sigma, n_sim, option_type='call', theta=None, seed=32):
    np.random.seed(seed)
    sign = 1 if option_type == 'call' else -1

    if theta is None:
        theta = (np.log(K / S) - (r - 0.5 * sigma**2) * T) / (sigma * np.sqrt(T)) # theta, formula (13.12)

    eps = np.random.standard_normal(n_sim) # Generate random value from N(theta, 1)
    Z = eps + theta # Generate random value from N(theta, 1)
    S_T = S * np.exp((r - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * Z)
    payoffs = np.maximum(sign * (S_T - K), 0)

    weights = np.exp(-theta * Z + 0.5 * theta**2) # likelihood ratio, formula (13.11)
    weighted_payoffs = payoffs * weights

    price = np.exp(-r * T) * weighted_payoffs.mean()
    se = np.exp(-r * T) * weighted_payoffs.std(ddof=1) / np.sqrt(n_sim)
    return price, se, theta

In [15]:
# Parameters
S, K, T, r, sigma, n_sim = 100.0, 200.0, 1.0, 0.05, 0.20, 10000

In [16]:
# Pricing
bs_price = black_scholes(S, K, T, r, sigma, 'call')
price_std, se_std = european_mc(S, K, T, r, sigma, n_sim, 'call', 16)
price_is, se_is, theta = european_mc_is(S, K, T, r, sigma, n_sim, 'call')

print(f"Black-Scholes             : {bs_price:.4f}")
print(f"Standard MC   (n = {n_sim:}) : {price_std:.4f} ± {1.96*se_std:.4f}")
print(f"Importance S. (n = {n_sim:}) : {price_is:.4f} ± {1.96*se_is:.4f}   (theta={theta:.4f})")
print(f"Variance reduction factor : {(se_std/se_is)**2:.0f}x")
print(se_std)

Black-Scholes             : 0.0048
Standard MC   (n = 10000) : 0.0086 ± 0.0077
Importance S. (n = 10000) : 0.0048 ± 0.0001   (theta=3.3157)
Variance reduction factor : 3755x
0.00392627227661009


### Quasi-Monte Carlo

In [12]:
def european_qmc_sobol(S, K, T, r, sigma, m_power, option_type='call', seed=32):
    sign = 1 if option_type == 'call' else -1

    sampler = qmc.Sobol(d=1, scramble=True, seed=seed)
    U = sampler.random_base2(m=m_power)
    Z = norm.ppf(U[:, 0])

    S_T = S * np.exp((r - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * Z)
    payoffs = np.maximum(sign * (S_T - K), 0)
    
    price = np.exp(-r * T) * payoffs.mean()
    return price, len(Z)

In [13]:
# Parameters
S, K, T, r, sigma, n_sim = 100.0, 100.0, 1.0, 0.05, 0.20, 10000

In [20]:
# Pricing
bs_price = black_scholes(100, 100, 1.0, 0.05, 0.20, 'call')
for m_power in [6, 8, 10, 12, 14]:
    n_sim = 2**m_power
    price_std, se_std = european_mc(100, 100, 1.0, 0.05, 0.20, n_sim, 'call')
    price_qmc, n_actual = european_qmc_sobol(100, 100, 1.0, 0.05, 0.20, m_power, 'call')

    print(f"n={n_sim:6} | Standard MC: {price_std:.4f} (err={abs(price_std-bs_price):.4f}) | "
          f"Sobol QMC: {price_qmc:.4f} (err={abs(price_qmc-bs_price):.4f})")

n=    64 | Standard MC: 11.4010 (err=0.9504) | Sobol QMC: 10.4164 (err=0.0342)
n=   256 | Standard MC: 12.1976 (err=1.7470) | Sobol QMC: 10.4330 (err=0.0176)
n=  1024 | Standard MC: 10.7552 (err=0.3046) | Sobol QMC: 10.4487 (err=0.0019)
n=  4096 | Standard MC: 10.7370 (err=0.2864) | Sobol QMC: 10.4494 (err=0.0012)
n= 16384 | Standard MC: 10.7579 (err=0.3073) | Sobol QMC: 10.4497 (err=0.0009)


In [23]:
def european_rqmc_sobol(S, K, T, r, sigma, m_power, n_replications=20, option_type='call'):
    sign = 1 if option_type == 'call' else -1
    prices = []
    for rep in range(n_replications):
        sampler = qmc.Sobol(d=1, scramble=True, seed=rep)
        U = sampler.random_base2(m=m_power)
        Z = norm.ppf(U[:, 0])
        S_T = S * np.exp((r - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * Z)
        payoffs = np.maximum(sign * (S_T - K), 0)
        prices.append(np.exp(-r * T) * payoffs.mean())

    prices = np.array(prices)
    price = prices.mean()
    se = prices.std(ddof=1) / np.sqrt(n_replications)
    return price, se

In [33]:
# Pricing
n_reps, m_power = 10, 10
n_total = n_reps * 2**m_power
price_std, se_std = european_mc(100, 100, 1.0, 0.05, 0.20, n_total, 'call')
price_rqmc, se_rqmc = european_rqmc_sobol(100, 100, 1.0, 0.05, 0.20, m_power, n_reps, 'call')

print(f"Black-Scholes                : {bs_price:.4f}")
print(f"Standard MC (n={n_total:})        : {price_std:.4f} ± {1.96*se_std:.4f}")
print(f"RQMC Sobol  ({n_reps} reps × {2**m_power:}) : {price_rqmc:.4f} ± {1.96*se_rqmc:.4f}")
print(f"Variance reduction factor    : {(se_std/se_rqmc)**2:.0f}x")

Black-Scholes                : 10.4506
Standard MC (n=10240)        : 10.6421 ± 0.2871
RQMC Sobol  (10 reps × 1024) : 10.4503 ± 0.0039
Variance reduction factor    : 5399x
